# Chapter 2: Working With Text Data

### Section 2.2 Tokenizing Text

The text we will tokenize for LLM training is “The Verdict,” a short story by Edith Wharton, which has been released into the public domain and is thus permitted to be used for LLM training tasks.

In [6]:
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
    "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x7c17a831efc0>)

In [7]:
import re

with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [8]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))
print(preprocessed[:30])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


### Section 2.3 Converting tokens into token IDs

To map the previously generated tokens into token IDs, we have to build a vocabulary
first. This vocabulary defines how we map each unique word and special character
to a unique integer.

Now that we have tokenized Edith Wharton’s short story and assigned it to a Python
variable called preprocessed, let’s create a list of all unique tokens and sort them
alphabetically to determine the vocabulary size:

In [9]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


After determining that the vocabulary size is 1,130 via the above code, we create the vocabulary
and print its first 51 entries for illustration purposes.

In [10]:
# upgraded this entire block in section 2.4

# these lines are added in section 2.4
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

#vocab: dict[int, str] = {token:integer for integer,token in enumerate(all_words)}
vocab: dict[int, str] = {token:integer for integer,token in enumerate(all_tokens)}  # section 2.4 upgrade
print(len(vocab.items()))
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

#for i, item in enumerate(vocab.items()):
#    print(item)
#    if i >= 50:
#        break

1132
('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


As we can see, the dictionary contains individual tokens associated with unique integer labels. Our next goal is to apply this vocabulary to convert new text into token IDs.

Based on the code output, we can confirm that the two new special tokens were indeed successfully incorporated into the vocabulary (section 2.4)

When we want to convert the outputs of an LLM from numbers back into text, we need a way to turn token IDs into text. For this, we can create an inverse version of the vocabulary that maps token IDs back to the corresponding text tokens.

Let’s implement a complete tokenizer class in Python with an encode method that splits text into tokens and carries out the string-to-integer mapping to produce token IDs via the vocabulary. In addition, we’ll implement a decode method that carries out the reverse integer-to-string mapping to convert the token IDs back into text. The following listing shows the code for this tokenizer implementation.

In [11]:
class SimpleTokenizerV1:
    def __init__(self, vocab: dict[str, int]) -> None:
        self.str_to_int: dict[str, int] = vocab
        self.int_to_str: dict[int, str] = {i:s for s,i in vocab.items()}

    def encode(self, text: str) -> list[int]:
        # Split text into words, punctuation marks, and whitespace separators.
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        # Drop empty strings and whitespace-only tokens before looking up IDs.
        preprocessed: list[str] = [ item.strip() for item in preprocessed if item.strip() ]
        ids: list[int] = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids: list[int]) -> str:
        # Convert each token ID back into its string token and join with spaces.
        text = " ".join([self.int_to_str[i] for i in ids])
        # Remove spaces that were inserted before punctuation during joining.
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [12]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [13]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


### Section 2.4 Adding Special Context Tokens

We need to modify the tokenizer to handle unknown words. We also need to address the usage and addition of special context tokens that can enhance a model’s understanding of context or other relevant information in the text. These special tokens can include markers for unknown words and document boundaries, for example. In particular, we will modify the vocabulary and tokenizer, SimpleTokenizerV2, to support two new tokens, <|unk|> and <|endoftext|>.

We added the special context chars to the vocabulary in previous cells. Below we implement the new tokenizer object.

In [14]:
class SimpleTokenizerV2:

    def __init__(self, vocab: dict[str, int]) -> None:
        self.str_to_int: dict[str, int] = vocab
        self.int_to_str: dict[int, str] = { i:s for s,i in vocab.items()}

    def encode(self, text: str) -> list[int]:
        preprocessed: list[str] = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [ item.strip() for item in preprocessed if item.strip() ]
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids: list[int] = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids: list[int]) -> str:
        text: str = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [15]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [16]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [17]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.
